# Limpieza y unión de fuentes

Juntamos las 11 tablas ya limpias (`DF_*_LIMPIO.csv` de buscametas, championsxip, carreirasgalegas, ccnorte, cronofinisher, cronorunner, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta) en una sola tabla, deduplicamos, derivamos las variables que hacen falta para el análisis y el modelo (año, mes, estación, `total_finishers`, `es_finde`, cruce con la población del INE, índice de carrera popular) y guardamos el resultado en dos ficheros:

- `DF_TODAS.csv`: solo las 12 columnas comunes a las 11 fuentes (`fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`), para quien quiera partir de cero y derivar sus propias columnas — algunos notebooks (p.ej. `Modelo_finishers.ipynb`) ya parten de este fichero.
- `DF_TODAS_UNION.csv`: la misma tabla con todo lo derivado ya calculado, pensada para `Analisis_union.ipynb` y para cualquier notebook de modelado que quiera partir de datos ya listos en vez de recalcularlos.

In [ ]:
from pathlib import Path
import pandas as pd
import re

FUENTES = {
    "buscametas": r"../../data/processed/buscametas/DF_BUSCAMETAS_LIMPIO.csv",
    "championsxip": r"../../data/processed/championsxip/DF_CHAMPIONSXIP_LIMPIO.csv",
    "carreirasgalegas": r"../../data/processed/carreirasgalegas/DF_CARREIRASGALEGAS_LIMPIO.csv",
    "ccnorte": r"../../data/processed/ccnorte/DF_CCNORTE_LIMPIO.csv",
    "cronofinisher": r"../../data/processed/cronofinisher/DF_CRONOFINISHER_LIMPIO.csv",
    "cronorunner": r"../../data/processed/cronorunner/DF_CRONORUNNER_LIMPIO.csv",
    "mychip": r"../../data/processed/mychip/DF_MYCHIP_LIMPIO.csv",
    "sportmaniacs": r"../../data/processed/sportmaniacs/DF_SPORTMANIACS_LIMPIO.csv",
    "cursescat": r"../../data/processed/cursescat/DF_CURSESCAT_LIMPIO.csv",
    "iter5": r"../../data/processed/iter5/DF_ITER5_LIMPIO.csv",
    "cruzandolameta": r"../../data/processed/cruzandolameta/DF_CRUZANDOLAMETA_LIMPIO.csv",
}

COMUNES = [
    "fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad",
    "publico", "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
]

tablas = []
for nombre, ruta in FUENTES.items():
    df = pd.read_csv(ruta, encoding="utf-8-sig")[COMUNES]
    assert (df["fuente"] == nombre).all(), f"'{nombre}': hay filas con 'fuente' distinto, revisar columnas"
    print(f"{nombre}: {len(df)} filas")
    tablas.append(df)

todas = pd.concat(tablas, ignore_index=True)
todas["fecha"] = pd.to_datetime(todas["fecha"])

print()
print("Total filas:", len(todas))
todas.head()


In [ ]:
CCAA_CANONICAS = {
    "andalucia": "Andalucía",
    "aragon": "Aragón",
    "asturias": "Asturias",
    "canarias": "Canarias",
    "cantabria": "Cantabria",
    "castilla y leon": "Castilla y León",
    "castilla la mancha": "Castilla-La Mancha",
    "castilla-la mancha": "Castilla-La Mancha",
    "catalunya": "Cataluña",
    "cataluna": "Cataluña",
    "comunitat valenciana": "Comunidad Valenciana",
    "comunidad valenciana": "Comunidad Valenciana",
    "extremadura": "Extremadura",
    "galicia": "Galicia",
    "illes balears": "Illes Balears",
    "islas baleares": "Illes Balears",
    "la rioja": "La Rioja",
    "murcia": "Murcia",
    "navarra": "Navarra",
    "pais vasco": "País Vasco",
    "país vasco": "País Vasco",
    "comunidad de madrid": "Comunidad de Madrid",
    "madrid": "Comunidad de Madrid",
    "ceuta": "Ceuta",
    "melilla": "Melilla",
}

PROVINCIA_CANONICAS = {
    "a coruna": "A Coruña",
    "la coruna": "A Coruña",
    "coruna": "A Coruña",
    "alava": "Álava",
    "araba": "Álava",
    "albacete": "Albacete",
    "alicante": "Alicante",
    "alacant": "Alicante",
    "almeria": "Almería",
    "avila": "Ávila",
    "badajoz": "Badajoz",
    "baleares": "Illes Balears",
    "barcelona": "Barcelona",
    "burgos": "Burgos",
    "caceres": "Cáceres",
    "cadiz": "Cádiz",
    "cantabria": "Cantabria",
    "castellon": "Castellón",
    "castello": "Castellón",
    "ciudad real": "Ciudad Real",
    "cordoba": "Córdoba",
    "cuenca": "Cuenca",
    "girona": "Girona",
    "gerona": "Girona",
    "granada": "Granada",
    "guadalajara": "Guadalajara",
    "guipuzcoa": "Gipuzkoa",
    "gipuzkoa": "Gipuzkoa",
    "huelva": "Huelva",
    "huesca": "Huesca",
    "jaen": "Jaén",
    "leon": "León",
    "lleida": "Lleida",
    "lerida": "Lleida",
    "la rioja": "La Rioja",
    "lugo": "Lugo",
    "madrid": "Madrid",
    "malaga": "Málaga",
    "murcia": "Murcia",
    "navarra": "Navarra",
    "ourense": "Ourense",
    "orense": "Ourense",
    "asturias": "Asturias",
    "asturies": "Asturias",
    "palencia": "Palencia",
    "las palmas": "Las Palmas",
    "pontevedra": "Pontevedra",
    "salamanca": "Salamanca",
    "santa cruz de tenerife": "Santa Cruz de Tenerife",
    "segovia": "Segovia",
    "sevilla": "Sevilla",
    "soria": "Soria",
    "tarragona": "Tarragona",
    "teruel": "Teruel",
    "toledo": "Toledo",
    "valencia": "Valencia",
    "valles": "Valencia",
    "valladolid": "Valladolid",
    "vizcaya": "Vizcaya",
    "bizkaia": "Vizcaya",
    "zamora": "Zamora",
    "zaragoza": "Zaragoza",
    "ceuta": "Ceuta",
    "melilla": "Melilla",
}

def _normalizar_geo_key(valor):
    if pd.isna(valor):
        return None
    s = str(valor).strip()
    if not s:
        return None
    s = s.replace("-", " ")
    s = re.sub(r"\s+", " ", s)
    s = s.lower()
    return s


def _canonizar_valor(valor, mapping):
    if pd.isna(valor):
        return pd.NA
    s = str(valor).strip()
    if not s:
        return pd.NA
    key = _normalizar_geo_key(s)
    return mapping.get(key, s)


def normalitzar_ubicacions(df):
    out = df.copy()

    for col in ["municipio", "comarca", "provincia", "comunidad_autonoma"]:
        if col in out.columns:
            out[col] = out[col].apply(lambda x: None if pd.isna(x) else str(x).strip() or None)

    if "provincia" not in out.columns:
        out["provincia"] = pd.NA
    if "comunidad_autonoma" not in out.columns:
        out["comunidad_autonoma"] = pd.NA

    out["provincia"] = out["provincia"].apply(lambda x: _canonizar_valor(x, PROVINCIA_CANONICAS) if x is not None else pd.NA)
    out["comunidad_autonoma"] = out["comunidad_autonoma"].apply(lambda x: _canonizar_valor(x, CCAA_CANONICAS) if x is not None else pd.NA)

    provincia_key = out["provincia"].fillna("").astype(str).str.strip().str.lower()
    ccan_key = out["comunidad_autonoma"].fillna("").astype(str).str.strip().str.lower()

    mask_ccaa_en_provincia = provincia_key.isin(set(CCAA_CANONICAS.keys()))
    if mask_ccaa_en_provincia.any():
        out.loc[mask_ccaa_en_provincia, "comunidad_autonoma"] = out.loc[mask_ccaa_en_provincia, "provincia"]
        out.loc[mask_ccaa_en_provincia, "provincia"] = pd.NA

    out["provincia"] = out["provincia"].apply(lambda x: _canonizar_valor(x, PROVINCIA_CANONICAS) if x is not None else pd.NA)
    out["comunidad_autonoma"] = out["comunidad_autonoma"].apply(lambda x: _canonizar_valor(x, CCAA_CANONICAS) if x is not None else pd.NA)

    out["comunidad_autonoma"] = out["comunidad_autonoma"].replace("", pd.NA)
    out["provincia"] = out["provincia"].replace("", pd.NA)
    return out

print("Normalitzador de comunitats/províncies aplicat a la unió")

todas = normalitzar_ubicacions(todas)
print("Províncies no nules:", todas["provincia"].notna().sum())
print("Comunitats detectades:", todas["comunidad_autonoma"].notna().sum())


In [2]:
# Un mismo evento puede haber sido capturado por más de una fuente (p.ej.
# una carrera popular que sale tanto en sportmaniacs como en cronofinisher)
# — sin quitar esto, esas carreras se cuentan dos veces. Detectamos
# duplicados por nombre normalizado (sin acentos/mayúsculas/puntuación,
# para no fallar por "1ª" vs "I" o tildes) + fecha exacta. Cuando el mismo
# evento aparece en varias fuentes, nos quedamos solo con la fuente que
# tiene más finishers sumados para ese evento — normalmente porque la otra
# fuente solo registró el evento sin resultados (filas en 0) o con menos
# categorías capturadas.
import unicodedata


def _normalizar_nombre(texto):
    if pd.isna(texto):
        return None
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii").upper()
    return re.sub(r"[^A-Z0-9]+", " ", t).strip()


todas["_nombre_norm"] = todas["nombre_carrera"].apply(_normalizar_nombre)
todas["_total_tmp"] = todas["finisher_d"] + todas["finisher_h"]

resumen = todas.groupby(["_nombre_norm", "fecha", "fuente"])["_total_tmp"].sum().reset_index()
n_fuentes = resumen.groupby(["_nombre_norm", "fecha"])["fuente"].transform("nunique")
resumen_dup = resumen[n_fuentes > 1]

idx_ganador = resumen_dup.groupby(["_nombre_norm", "fecha"])["_total_tmp"].idxmax()
ganador = resumen_dup.loc[idx_ganador, ["_nombre_norm", "fecha", "fuente"]].rename(
    columns={"fuente": "_fuente_ganadora"}
)

antes = len(todas)
todas = todas.merge(ganador, on=["_nombre_norm", "fecha"], how="left")
es_evento_duplicado = todas["_fuente_ganadora"].notna()
filas_a_quitar = es_evento_duplicado & (todas["fuente"] != todas["_fuente_ganadora"])

print(f"Eventos duplicados entre fuentes: {ganador.shape[0]}")
print(f"Filas eliminadas (de la fuente no ganadora): {filas_a_quitar.sum()}")
print(f"  - con 0 finishers (fuente que no llegó a capturar resultados): "
      f"{(filas_a_quitar & (todas['_total_tmp'] == 0)).sum()}")
print(f"  - con datos, pero menos completos que la fuente ganadora: "
      f"{(filas_a_quitar & (todas['_total_tmp'] > 0)).sum()}")
print()
print("Filas eliminadas, por fuente:")
print(todas.loc[filas_a_quitar, "fuente"].value_counts())

todas = todas.loc[~filas_a_quitar].drop(columns=["_nombre_norm", "_total_tmp", "_fuente_ganadora"])
print()
print(f"Total filas: {antes} -> {len(todas)}")

Eventos duplicados entre fuentes: 29
Filas eliminadas (de la fuente no ganadora): 65
  - con 0 finishers (fuente que no llegó a capturar resultados): 37
  - con datos, pero menos completos que la fuente ganadora: 28

Filas eliminadas, por fuente:
fuente
cronofinisher     25
mychip            25
cruzandolameta    10
sportmaniacs       3
ccnorte            2
Name: count, dtype: int64

Total filas: 46053 -> 45988


In [3]:
# Además de eventos repetidos entre fuentes, hay filas totalmente
# idénticas dentro de una misma fuente (mismo evento, categoría y hasta el
# mismo número de finishers) — repartidas entre 7 fuentes distintas sin
# que ninguna concentre la mayoría, así que parece un artefacto general de
# scraping (páginas o categorías leídas dos veces) y no un fallo de una
# fuente en concreto. Al ser copias exactas, quedarnos con una sola no
# pierde ninguna información real.
cols_evento = [
    "fuente", "nombre_carrera", "fecha", "distancia", "tipo_modalidad", "publico",
    "finisher_d", "finisher_h",
]
antes = len(todas)
print("Filas idénticas dentro de la misma fuente, por fuente:")
print(todas.loc[todas.duplicated(subset=cols_evento, keep=False), "fuente"].value_counts())

todas = todas.drop_duplicates(subset=cols_evento).reset_index(drop=True)
print()
print(f"Total filas: {antes} -> {len(todas)}")

Filas idénticas dentro de la misma fuente, por fuente:
fuente
cronofinisher     668
sportmaniacs      608
ccnorte           433
mychip            329
championsxip      100
cruzandolameta     10
buscametas          2
Name: count, dtype: int64

Total filas: 45988 -> 44689


In [4]:
# cronofinisher (79% de sus filas) y carreirasgalegas (66%) tienen una
# proporción de filas en 0/0 mucho más alta que el resto de fuentes, y no
# se explica por categorías especiales (pasa igual en Absoluta/General) ni
# solo por carreras que aún no se han disputado (hay filas en 0 desde
# 2015/2019) — son categorías que existieron pero de las que nunca se
# llegó a capturar el resultado real. Las marcamos como NaN, no 0, igual
# que ya se hace con el centinela de "distancia" — así no se cuentan como
# "carrera real de 0 personas" ni en las estadísticas ni en un futuro modelo.
_FUENTES_SIN_RESULTADOS = ["cronofinisher", "carreirasgalegas"]
_sin_resultados = (
    todas["fuente"].isin(_FUENTES_SIN_RESULTADOS)
    & (todas["finisher_d"] == 0)
    & (todas["finisher_h"] == 0)
)

print(f"Filas marcadas como sin resultados (antes 0/0): {_sin_resultados.sum()}")
print(todas.loc[_sin_resultados, "fuente"].value_counts())

todas["finisher_d"] = todas["finisher_d"].astype("float64")
todas["finisher_h"] = todas["finisher_h"].astype("float64")
todas.loc[_sin_resultados, ["finisher_d", "finisher_h"]] = float("nan")

Filas marcadas como sin resultados (antes 0/0): 2435
fuente
carreirasgalegas    1858
cronofinisher        577
Name: count, dtype: int64


In [5]:
# La fusión "Cadete/Juvenil" -> "Infantil" ya no se hace aquí: se movió a
# cada Limpieza_<fuente>.ipynb (carreirasgalegas y ccnorte ya lo hacían así
# en origen; el resto se cambió para que 'Infantil' signifique lo mismo en
# las 11 fuentes desde su propia limpieza, no como parche al unir). Solo
# comprobamos aquí que ninguna fuente se haya quedado atrás.
assert "Cadete/Juvenil" not in todas["publico"].unique(), (
    "alguna fuente todavia distingue Cadete/Juvenil de Infantil - revisar su Limpieza_<fuente>.ipynb"
)
print(todas["publico"].value_counts())

publico
Absoluta/General     32009
Infantil             10741
Otros                  590
Mayores/Veteranos      519
Equipos                497
Elite                  333
Name: count, dtype: int64


In [6]:
# distancia / municipio / comarca / provincia pueden venir vacíos según la
# fuente; finisher_d/finisher_h también, pero solo en cronofinisher y
# carreirasgalegas (las filas sin resultados reales que acabamos de marcar
# como NaN) -- el resto de columnas no debería tener ningún nulo.
print("Filas por fuente:")
print(todas["fuente"].value_counts())
print()

print("Nulos por columna:")
print(todas.isna().sum())
print()

print("Rango de fechas:", todas["fecha"].min(), "->", todas["fecha"].max())

Filas por fuente:
fuente
sportmaniacs        20464
ccnorte              6023
mychip               4545
cronorunner          3145
carreirasgalegas     2797
championsxip         2580
iter5                1834
cruzandolameta       1677
cronofinisher         842
cursescat             469
buscametas            313
Name: count, dtype: int64

Nulos por columna:
fuente                0
nombre_carrera        0
fecha                 0
dia_semana            0
distancia             0
tipo_modalidad        0
publico               0
finisher_d         4037
finisher_h         4037
municipio          2475
comarca           20673
provincia          2696
dtype: int64

Rango de fechas: 2003-11-16 00:00:00 -> 2026-09-04 00:00:00


In [7]:
SALIDA = Path("../../data/processed/union/DF_TODAS.csv")
todas.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print("Guardado en", SALIDA)

Guardado en ../../data/processed/union/DF_TODAS.csv


## Variables derivadas y cruce con población

Antes de meternos en el análisis añadimos las columnas que vamos a necesitar todo el rato: año, mes y estación a partir de la fecha, el total de finishers por fila, si la carrera es de fin de semana, y la población del municipio del año de la carrera y su superficie/densidad (cruzando con la serie histórica del INE de `Limpieza_INE.ipynb`). Así las tenemos ya listas como variables numéricas para el univariable, la correlación y el cruce con los objetivos.

In [8]:
MES_A_ESTACION = {
    12: "Invierno", 1: "Invierno", 2: "Invierno",
    3: "Primavera", 4: "Primavera", 5: "Primavera",
    6: "Verano", 7: "Verano", 8: "Verano",
    9: "Otoño", 10: "Otoño", 11: "Otoño",
}

todas["anyo"] = todas["fecha"].dt.year
todas["mes"] = todas["fecha"].dt.month
todas["estacion"] = todas["mes"].map(MES_A_ESTACION)
todas["total_finishers"] = todas["finisher_d"] + todas["finisher_h"]
todas["es_finde"] = todas["dia_semana"].isin(["Sábado", "Domingo"])

todas[["fecha", "anyo", "mes", "estacion", "total_finishers", "es_finde"]].head()

,fecha,anyo,mes,estacion,total_finishers,es_finde
0,2026-07-25,2026,7,Verano,164.0,True
1,2026-07-25,2026,7,Verano,30.0,True
2,2026-07-25,2026,7,Verano,40.0,True
3,2026-07-25,2026,7,Verano,33.0,True
4,2026-07-25,2026,7,Verano,12.0,True


In [9]:
import glob
import re
import unicodedata

def _despostponer_articulo(texto_norm):
    """Pasa el artículo pospuesto entre paréntesis del nomenclátor ("CORUÑA (A)",
    "POBLA DE FARNALS (LA)") delante, que es como se escribe en las fuentes ("A
    CORUÑA", "LA POBLA DE FARNALS") -- sin espacio tras el apóstrofo elidido de
    "L'", con espacio para el resto de artículos. Recibe texto ya en mayúsculas
    sin acentos (de `normalizar()` o de `municipio_norm`/`nombre_ine` del INE).
    """
    if texto_norm is None:
        return None
    m = re.match(r"^(.*?)\s*\((L'|LA|EL|LOS|LAS|ELS|LES|A|O|AS|OS)\)$", texto_norm)
    if not m:
        return texto_norm
    nombre, articulo = m.groups()
    sep = "" if articulo.endswith("'") else " "
    return f"{articulo}{sep}{nombre}".strip()


def normalizar(texto):
    """Mayúsculas y sin acentos, para poder cruzar texto libre con el nomenclátor del INE.

    También normaliza dos formatos propios del nomenclátor: los nombres dobles
    oficiales tipo "ALBORAIA/ALBORAYA" (se homogeniza el espaciado alrededor de
    la barra) y el artículo pospuesto entre paréntesis tipo "ALCÚDIA (L')" o
    "CORUÑA (A)" (se pasa a "L'ALCUDIA"/"A CORUNA", que es como suele venir
    escrito en las fuentes — sin espacio tras el apóstrofo elidido de "L'",
    con espacio para el resto de artículos).
    """
    if pd.isna(texto):
        return None
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii").upper().strip()
    t = re.sub(r"\s*/\s*", "/", t)
    return _despostponer_articulo(t)

# Las comunidades multiprovinciales (Galicia, Catalunya, Andalucía, la
# Comunidad Valenciana, Castilla y León, Castilla-La Mancha, Aragón) no se
# pueden asignar a una sola provincia del INE con el nombre de la comunidad
# solo, así que además de la tabla de provincias guardamos, para cada una de
# estas comunidades, las provincias que la componen: si el municipio tiene un
# nombre único dentro de esas provincias, podemos igualmente encontrar su
# población sin ambigüedad (ver el fallback más abajo).
PROVINCIA_INE = {
    1: ["Álava", "Araba", "Araba/Álava"], 2: ["Albacete"], 3: ["Alicante", "Alacant"],
    4: ["Almería"], 5: ["Ávila"], 6: ["Badajoz"], 7: ["Baleares", "Illes Balears", "Baleares (Illes)"],
    8: ["Barcelona"], 9: ["Burgos"], 10: ["Cáceres"], 11: ["Cádiz"], 12: ["Castellón", "Castelló"],
    13: ["Ciudad Real"], 14: ["Córdoba"], 15: ["La Coruña", "A Coruña"], 16: ["Cuenca"],
    17: ["Girona", "Gerona"], 18: ["Granada"], 19: ["Guadalajara"], 20: ["Guipúzcoa", "Gipuzkoa"],
    21: ["Huelva"], 22: ["Huesca"], 23: ["Jaén"], 24: ["León"], 25: ["Lleida", "Lérida"],
    26: ["La Rioja"], 27: ["Lugo"], 28: ["Madrid", "Comunidad de Madrid"], 29: ["Málaga"],
    30: ["Murcia", "Región de Murcia"], 31: ["Navarra"], 32: ["Ourense", "Orense"],
    33: ["Asturias", "Asturies", "Asturias / Asturies"], 34: ["Palencia"], 35: ["Las Palmas"],
    36: ["Pontevedra"], 37: ["Salamanca"], 38: ["Santa Cruz de Tenerife"], 39: ["Cantabria"],
    40: ["Segovia"], 41: ["Sevilla"], 42: ["Soria"], 43: ["Tarragona"], 44: ["Teruel"],
    45: ["Toledo"], 46: ["Valencia", "València"], 47: ["Valladolid"], 48: ["Vizcaya", "Bizkaia"],
    49: ["Zamora"], 50: ["Zaragoza"], 51: ["Ceuta"], 52: ["Melilla"],
}
PROVINCIA_A_CODIGO = {normalizar(n): c for c, ns in PROVINCIA_INE.items() for n in ns}

CCAA_MULTIPROVINCIA = {
    "Galicia": [15, 27, 32, 36],
    "Catalunya": [8, 17, 25, 43],
    "Cataluña": [8, 17, 25, 43],
    "Andalucía": [4, 11, 14, 18, 21, 23, 29, 41],
    "Comunitat Valenciana": [3, 12, 46],
    "Comunidad Valenciana": [3, 12, 46],
    "País Vasco": [1, 20, 48],
    "Aragón": [22, 44, 50],
    "Castilla-La Mancha": [2, 13, 16, 19, 45],
    "Castile-La Mancha": [2, 13, 16, 19, 45],
    "Castille-La Mancha": [2, 13, 16, 19, 45],
    "Castilla y León": [5, 9, 24, 34, 37, 40, 42, 47, 49],
    "Extremadura": [6, 10],
    "Canarias": [35, 38],
}
CCAA_A_PROVINCIAS = {normalizar(k): v for k, v in CCAA_MULTIPROVINCIA.items()}

# --- Población y superficie: ahora vienen de Limpieza_INE.ipynb -----------
# DF_INE_POBLACION_HISTORICO.csv trae una fila por (municipio, año) en vez de
# un único año fijo como el DF_INE_Poblaciones_España.xlsx anterior, así que
# el cruce pasa a hacerse por (municipio_norm, provincia_ine, año). Para
# carreras de años sin padrón todavía (el año en curso) o anteriores al primer
# año disponible, usamos el año disponible más cercano -- mismo criterio de
# "quedarnos con el último año disponible" que ya veníamos aplicando en otros
# sitios del proyecto. La superficie no depende del año (los límites
# municipales apenas cambian), así que se cruza aparte y sirve para la
# densidad de población.
INE_DIR = Path("../../data/processed/INE")
ine_historico = pd.read_csv(INE_DIR / "DF_INE_POBLACION_HISTORICO.csv")
ine_superficie = pd.read_csv(INE_DIR / "DF_INE_SUPERFICIE_MUNICIPIOS.csv")

# `Limpieza_INE.ipynb` normaliza mayúsculas/acentos pero no despospone el
# artículo entre paréntesis del nomenclátor (esa conversión se añadió después,
# aquí, solo para el lado de `todas`) -- sin este paso ~15.500 filas del INE se
# quedan con nombres tipo "POBLA DE FARNALS (LA)" que nunca cruzan contra
# "LA POBLA DE FARNALS", la forma en que vienen las fuentes. Se aplica antes de
# expandir los alias dobles porque algún nombre combina ambos formatos (p.ej.
# "BENITACHELL/POBLE NOU DE BENITATXELL (EL)").
ine_historico["municipio_norm"] = ine_historico["municipio_norm"].apply(_despostponer_articulo)
ine_superficie["municipio_norm"] = ine_superficie["municipio_norm"].apply(_despostponer_articulo)

ANYO_MIN_DISPONIBLE = ine_historico["anyo"].min()
ANYO_MAX_DISPONIBLE = ine_historico["anyo"].max()

# El INE usa nombres dobles oficiales tipo "ALBORAIA/ALBORAYA"; los añadimos
# también como dos filas independientes (una por cada mitad) para poder
# cruzar aunque una fuente solo use una de las dos formas del nombre. Se
# aplica igual a población (por año) y a superficie (una sola fila).
def expandir_alias_dobles(df):
    dobles = df[df["municipio_norm"].str.contains("/", na=False)]
    if dobles.empty:
        return df
    alias = [
        {**fila.to_dict(), "municipio_norm": mitad.strip()}
        for _, fila in dobles.iterrows()
        for mitad in fila["municipio_norm"].split("/")
    ]
    return pd.concat([df, pd.DataFrame(alias)], ignore_index=True)

ine_historico = expandir_alias_dobles(ine_historico)
ine_historico = ine_historico.drop_duplicates(subset=["municipio_norm", "provincia_ine", "anyo"], keep="first")

ine_superficie = expandir_alias_dobles(ine_superficie)
ine_superficie = ine_superficie.drop_duplicates(subset=["municipio_norm", "provincia_ine"], keep="first")

todas["municipio_norm"] = todas["municipio"].apply(normalizar)
todas["provincia_norm"] = todas["provincia"].apply(normalizar)
def resolver_provincia_directa(provincia_norm):
    """Prueba el nombre completo y, si trae barra ("VALENCIA/VALENCIA",
    "ASTURIES" suelto no la tiene pero "ALACANT/ALICANTE" sí), cada mitad
    por separado contra el diccionario de provincias."""
    if provincia_norm is None:
        return None
    for parte in provincia_norm.split("/"):
        if parte in PROVINCIA_A_CODIGO:
            return PROVINCIA_A_CODIGO[parte]
    return None


todas["provincia_ine"] = todas["provincia_norm"].apply(resolver_provincia_directa)

# Año de cruce: el de la propia carrera, salvo que caiga fuera del rango que
# cubre el INE -- se acerca al extremo disponible más próximo en vez de dejarlo
# sin cruzar.
todas["anyo_cruce"] = todas["anyo"].clip(lower=ANYO_MIN_DISPONIBLE, upper=ANYO_MAX_DISPONIBLE)
n_ajustados = int((todas["anyo_cruce"] != todas["anyo"]).sum())
print(
    f"Filas cuyo año de carrera queda fuera de {ANYO_MIN_DISPONIBLE}-{ANYO_MAX_DISPONIBLE} "
    f"(se cruza con el año disponible más cercano): {n_ajustados}"
)

todas = todas.merge(
    ine_historico[["municipio_norm", "provincia_ine", "anyo", "total_poblacion", "poblacion_h", "poblacion_d"]],
    left_on=["municipio_norm", "provincia_ine", "anyo_cruce"],
    right_on=["municipio_norm", "provincia_ine", "anyo"],
    how="left",
    suffixes=("", "_ine"),
)
todas = todas.drop(columns=["anyo_ine"], errors="ignore")

todas = todas.merge(
    ine_superficie[["municipio_norm", "provincia_ine", "superficie_km2"]],
    on=["municipio_norm", "provincia_ine"],
    how="left",
)
todas["densidad_poblacion"] = todas["total_poblacion"] / todas["superficie_km2"]

# Fallback para comunidades multiprovinciales: cuando 'provincia' es en
# realidad el nombre de una comunidad autónoma con varias provincias (o el
# cruce anterior no encontró nada), buscamos el municipio dentro de las
# provincias de esa comunidad, para el año de cruce correspondiente. Solo lo
# damos por válido si el nombre del municipio es único en toda la comunidad
# ese año, para no arriesgarnos a cruzar con la provincia equivocada cuando
# el nombre se repite.
_lookup = {}
for (municipio, anyo_fila), grupo in ine_historico.groupby(["municipio_norm", "anyo"]):
    for _, fila in grupo.iterrows():
        _lookup.setdefault(municipio, {}).setdefault(anyo_fila, {})[fila["provincia_ine"]] = fila

_superficie_lookup = {}
for _, fila in ine_superficie.iterrows():
    _superficie_lookup.setdefault(fila["municipio_norm"], {})[fila["provincia_ine"]] = fila["superficie_km2"]

_pendientes = todas.index[todas["total_poblacion"].isna()]
_resueltos_ccaa = 0
def candidatas_ccaa(provincia_norm):
    if provincia_norm is None:
        return None
    encontradas = set()
    for parte in provincia_norm.split("/"):
        encontradas.update(CCAA_A_PROVINCIAS.get(parte, []))
    return list(encontradas) if encontradas else None


for idx in _pendientes:
    candidatas = candidatas_ccaa(todas.at[idx, "provincia_norm"])
    if not candidatas:
        continue
    anyo_fila = todas.at[idx, "anyo_cruce"]
    opciones = _lookup.get(todas.at[idx, "municipio_norm"], {}).get(anyo_fila)
    if not opciones:
        continue
    encontrados = {p: v for p, v in opciones.items() if p in candidatas}
    if len(encontrados) == 1:
        prov_ine, fila = list(encontrados.items())[0]
        todas.loc[idx, ["provincia_ine", "total_poblacion", "poblacion_h", "poblacion_d"]] = [
            prov_ine, fila["total_poblacion"], fila["poblacion_h"], fila["poblacion_d"]
        ]
        superficie_val = _superficie_lookup.get(todas.at[idx, "municipio_norm"], {}).get(prov_ine)
        if superficie_val is not None:
            todas.loc[idx, "superficie_km2"] = superficie_val
            todas.loc[idx, "densidad_poblacion"] = fila["total_poblacion"] / superficie_val
        _resueltos_ccaa += 1

cobertura = 100 * todas["total_poblacion"].notna().mean()
print(f"Filas resueltas gracias al fallback de comunidad autónoma: {_resueltos_ccaa}")
print(f"Filas con población cruzada: {todas['total_poblacion'].notna().sum()} de {len(todas)} ({cobertura:.1f}%)")
print(f"Filas con superficie/densidad cruzada: {todas['superficie_km2'].notna().sum()} de {len(todas)}")

Filas cuyo año de carrera queda fuera de 2000-2025 (se cruza con el año disponible más cercano): 2295
Filas resueltas gracias al fallback de comunidad autónoma: 5796
Filas con población cruzada: 38236 de 44689 (85.6%)
Filas con superficie/densidad cruzada: 38230 de 44689


In [10]:
print("Cobertura del cruce con el INE por fuente (% de filas con población encontrada):")
print(todas.groupby("fuente")["total_poblacion"].apply(lambda s: round(100 * s.notna().mean(), 1)))
print()

print("Valores de 'provincia' sin equivalente en el INE (top 10, casi todos son comunidades autónomas):")
print(todas.loc[todas["provincia"].notna() & todas["provincia_ine"].isna(), "provincia"].value_counts().head(10))

Cobertura del cruce con el INE por fuente (% de filas con población encontrada):
fuente
buscametas          72.5
carreirasgalegas    83.8
ccnorte             64.4
championsxip        82.4
cronofinisher       89.9
cronorunner         85.9
cruzandolameta      91.6
cursescat           89.6
iter5               72.3
mychip              75.0
sportmaniacs        95.4
Name: total_poblacion, dtype: float64

Valores de 'provincia' sin equivalente en el INE (top 10, casi todos son comunidades autónomas):
provincia
Galicia                      515
Catalunya                    137
Aragón                        54
Comunitat Valenciana          41
Andalucía                     36
Cualquier lugar del mundo      9
País Vasco                     6
Comunidad Valenciana           5
Cataluña                       1
ESPAÑA                         1
Name: count, dtype: int64


Antes de mejorar el cruce, la cobertura era baja sobre todo en `ccnorte` (27.7%), `mychip` (52.5%) y `championsxip` (63.4%), casi siempre por el mismo motivo: la columna `provincia` guarda en realidad el nombre de la comunidad autónoma (Galicia, Catalunya, Comunidad Valenciana...), y una comunidad con varias provincias no se puede asignar directamente a un código INE sin más información. Añadimos cuatro mejoras al cruce (celda de arriba): (1) cuando `provincia` es una comunidad multiprovincial, buscamos el municipio dentro de las provincias que la componen y solo lo damos por válido si el nombre es único en toda la comunidad, para no arriesgarnos a cruzar con la provincia equivocada; (2) tratamos como equivalentes las dos mitades de los nombres oficiales dobles del nomenclátor, tanto en el municipio (p.ej. "Alboraia/Alboraya") como en la propia `provincia` (p.ej. "València / Valencia", "Alacant / Alicante"); (3) el nomenclátor pospone el artículo entre paréntesis ("CORUÑA (A)", "PINO (O)", "ALCÚDIA (L')") — lo pasamos delante, que es como se escribe en las fuentes ("A Coruña", "O Pino", "l'Alcúdia"), cuidando de no meter un espacio de más cuando el artículo lleva apóstrofo elidido; y (4) esa misma conversión del artículo pospuesto, que al principio solo se aplicaba al nombre de las carreras, ahora se aplica también a `municipio_norm` del lado del INE (`ine_historico`/`ine_superficie`) — sin esto, ~15.500 filas del nomenclátor se quedaban con el nombre en su forma original ("POBLA DE FARNALS (LA)") y nunca cruzaban contra la forma con el artículo delante que traen las fuentes, sobre todo en la Comunidad Valenciana, donde este formato es muy común. Con esto la cobertura global sube del ~73% al 85.6%; en concreto `ccnorte` pasa a 64.4%, `mychip` a 75.0%, `championsxip` a 82.4% y `sportmaniacs` (el más beneficiado por el punto 4, con la mayoría de nombres tipo "(LA)"/"(EL)" en su catálogo) a 95.4%. El resto de la brecha es sobre todo el mismo municipio escrito con nombres distintos: exónimos históricos en castellano frente al nombre oficial actual ("Alcira" vs "Alzira", "Játiva" vs "Xàtiva" en mychip), nombres truncados y ambiguos entre varios municipios de una misma comunidad (p.ej. "Sant Llorenç" a secas, que coincide con 4 municipios distintos de Catalunya), municipios sin el artículo que sí lleva su nombre oficial (p.ej. "Pobla de Vallbona" por "La Pobla de Vallbona"), entidades locales menores que no son un municipio propio en el nomenclátor (p.ej. "Mareny de Barraquetes", pedanía de Sueca), y al menos un caso de `provincia` mal etiquetada en origen (carreras de Barbastro, Huesca, marcadas como `Catalunya` en `iter5`). Cerrar esto ya requeriría una tabla de sinónimos hecha a mano o revisar la fuente caso a caso — queda como posible ampliación futura.

`cronorunner` llega aquí con `municipio`/`comarca`/`provincia` ya geocodificados de verdad en `Limpieza_cronorunner.ipynb` (con Nominatim, sobre el municipio real cuando lo hay y si no sobre un candidato extraído del nombre del evento), así que cruza con el INE igual de bien que el resto: 85.9% de cobertura, por encima incluso de la media global. El hueco que queda es la misma causa que ya limita el propio `municipio` de origen: en 1.605 de sus 3.145 filas (51%) el scraper marcó `municipio` como `"Sin Localidad definida"` porque la ficha del evento no traía icono de ubicación, y ahí la geocodificación por candidato de nombre no siempre encuentra nada real que cruzar (149 de las 754 consultas únicas a Nominatim, entre municipios reales y candidatos, se quedaron sin resultado). A partir de aquí, cualquier cálculo con `total_poblacion` trabaja solo sobre ese subconjunto cruzado, no sobre la tabla entera.

### `es_carrera_popular`: identificar carreras populares por el nombre

Las carreras populares ("carrera/cursa/carreira popular") son un tipo de evento que se reconoce por su propio nombre, en las 3 lenguas de las fuentes: pensadas para participación masiva y abierta a todo el mundo, no para competición federada. Además de la palabra "popular" en sí, ampliamos la detección a otros patrones de nombre igual de reconocibles y validados contra los datos antes de incluirlos:

- **"San Silvestre"/"Sant Silvestre"**: el formato más estandarizado que hay — carrera de Nochevieja abierta a todo el mundo, prácticamente siempre popular por definición. Se amplía para cubrir también la grafía catalana (`Sant Silvestre de Berga`, `Sant Silvestre del Masnou`...), que antes se quedaba fuera.
- **"solidari(a/o)"**: carreras benéficas/solidarias — abiertas y participativas, no competitivas.
- **"festa/festes/fiesta"** (con límite de palabra, para no capturar "manifestación" y similares): carreras de fiesta mayor/patronal.
- **"nocturn[oa]"/"night"**: carreras nocturnas, en las 2 lenguas y también en inglés (`Night Race`, `Urban Trail Night`...) — el patrón anterior solo cogía "nocturna" en femenino y se dejaba fuera "nocturno" y los nombres en inglés.
- **"mujer"/"dona"**: carreras de la mujer/cursa de la dona — eventos participativos abiertos, casi siempre en categoría Absoluta/General.
- **"milla urbana"**: formato clásico de evento popular de ciudad, sobre todo en categorías infantiles.
- **"legua"**: unidad de distancia tradicional que da nombre a carreras populares de pueblo (`Legua Irmandiña`, `Legua Ciudad de Valladolid`...).

Además, el patrón se aplica ahora sobre el nombre **sin acentos** (reutilizando `_normalizar_nombre`, ya definida más arriba para el cruce de duplicados) en vez de sobre el nombre tal cual: el patrón `solidari` no estaba pillando la grafía catalana `SOLIDÀRIA` por la tilde, y esa sola diferencia dejaba fuera 242 filas que sí eran carreras solidarias reales.

Se descartaron `san`/`sant` a secas (demasiado ruidoso: mezcla carreras patronales populares con pruebas serias tituladas "San/Sant `<algo>`", el mismo problema por el que ya se había descartado "memorial") y `trofeo`/`nacional`/`circuito`/`premio` (van justo en la dirección contraria: son señal de evento competitivo/federado, no de carrera popular).

Con todos los patrones nuevos, el índice pasa de 7.464 a 9.902 filas (18.0% → 23.8% del total), y sigue encajando con el patrón esperado: sobre todo `road running`, y `publico` Absoluta/General o Infantil.

In [11]:
PATRON_POPULAR = (
    r"\bpopular|\bsant? silvestre\b|\bsolidari|\bfiesta\b|\bfestes?\b"
    r"|\bnocturn[oa]\b|\bnight\b|\blegua\b|\bmujer\b|\bdona\b|\bmilla urbana\b"
)
_nombre_sin_acentos = todas["nombre_carrera"].apply(_normalizar_nombre)
todas["es_carrera_popular"] = _nombre_sin_acentos.str.contains(PATRON_POPULAR, case=False, na=False, regex=True)

print(f"Carreras populares detectadas: {todas['es_carrera_popular'].sum()} de {len(todas)} ({100 * todas['es_carrera_popular'].mean():.1f}%)")
print()
print("Por fuente:")
print(todas.loc[todas["es_carrera_popular"], "fuente"].value_counts())
print()
print("Por tipo_modalidad:")
print(todas.loc[todas["es_carrera_popular"], "tipo_modalidad"].value_counts())
print()
print("Por publico:")
print(todas.loc[todas["es_carrera_popular"], "publico"].value_counts())

Carreras populares detectadas: 10450 de 44689 (23.4%)

Por fuente:
fuente
sportmaniacs        4328
ccnorte             2077
carreirasgalegas    1303
mychip               761
cronorunner          548
cruzandolameta       530
championsxip         471
cronofinisher        259
iter5                103
cursescat             50
buscametas            20
Name: count, dtype: int64

Por tipo_modalidad:
tipo_modalidad
road running       8398
Otros              1172
trail running       533
marcha              227
Multidisciplina      63
Ciclismo y btt       57
Name: count, dtype: int64

Por publico:
publico
Absoluta/General     6417
Infantil             3680
Otros                 168
Mayores/Veteranos     135
Equipos                37
Elite                  13
Name: count, dtype: int64


## Guardado final

Guardamos la tabla ya unida, con las variables derivadas, el cruce con población/superficie/densidad por año y el índice de carrera popular, lista para `Analisis_union.ipynb` y para cualquier futuro notebook de modelado. `DF_TODAS.csv` (guardado más arriba, sin estas columnas) se mantiene igual por compatibilidad con los notebooks que ya lo usan directamente.

In [12]:
SALIDA_UNION = Path("../../data/processed/union/DF_TODAS_UNION.csv")
todas.to_csv(SALIDA_UNION, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA_UNION}")
print("Columnas:", list(todas.columns))

Guardado en ../../data/processed/union/DF_TODAS_UNION.csv
Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'anyo', 'mes', 'estacion', 'total_finishers', 'es_finde', 'municipio_norm', 'provincia_norm', 'provincia_ine', 'anyo_cruce', 'total_poblacion', 'poblacion_h', 'poblacion_d', 'superficie_km2', 'densidad_poblacion', 'es_carrera_popular']
